# Extraction and Preprocessing 

## What it does: 
Processes raw precise point positioning data from GPS position files to calculate daily ice velocity estimates for the GPS stations. It uses this velcotiy to create a velocity time-series which allows every icequake to have velocity as a feature. 

## Inputs: 
- PPP files 
- Slip event catalog 

## Outputs: 
- Event catalog that gives the pre-event velocity median for each event, the velocity components per station, and how many observations were used 
- Daily velocity time series 

## References: 
- Katz, Z. S., Siegfried, M. R., & Padman, L. (2026). Slip-event timing and ice velocity vary at long-period ocean tidal frequencies at Whillans Ice Plain, West Antarctica. Journal of Geophysical Research: Earth Surface, 131, e2025JF008770. https://doi.org/10.1029/2025JF008770
- https://zenodo.org/records/17797751 
- https://github.com/zsk4/WhillansCatalogPaper/releases/tag/v1.2.1 



In [1]:
import os, glob
import numpy as np
import pandas as pd
from pyproj import Transformer

# path configuration 

PPP_dir = "../Data/PPP/PPP/"
Events_csv = "../Data/whillans_velocity_per_event.csv"
Daily_timeseries_csv = "../Data/ppp_velocity_timeseries.csv"
Event_w_vel_csv = "../Data/whillans_events_with_velocity.csv"
Pre_filter_csv = "../Data/ppp_pre_filter_epochs.csv"

#cut offs for preprocessing/filtering the data 
phys_vel_threshold = 2000    # drop computed velocity that is higher than this (physical upper bound for Antarctic ice streams)
sec_per_year = 365.25 * 86400 # converting m/s to m/yr
min_epochs_day = 1000    # setting a limit for how many epochs are needed to give a reasonable median (this is about 17% so a low threshold)
max_day_gap = 2       # max days between consecutive daily estimates
pre_window_days = 30      # days before event to average for pre-event velocity
min_daily_pts = 5       # minimum daily estimates in pre-event window

# Cooridnate Projection
# lat/lon to EPSG:3031 (Antarctic polar sterographic coordinates) so that velocity components are in meters 
geo_to_polar = Transformer.from_crs("EPSG:4326", "EPSG:3031", always_xy=True)

def proj_geo_to_polar(lat, lon):
    east, north = geo_to_polar.transform(lon, lat)
    return east, north


# Looked at what the columns were using describe in command line to set these 

column_labels = [
    "dir", "frame", "stn", "doy", "date", "time", #direction, reference frame, station ID, day of year, date (yyyy-mm-dd), time
    "nsv", "gdop", "rmsc", "rmsp", # number of satellietes used, geometric effects on precision, RMS of code observations, RMS of phase observations
    "dlat", "dlon", "dhgt", # displacement from reference latitude, longitude and height
    "sdlat", "sdlon", "sdhgt", # 95% confidence for lat, long, and height
    "latdd", "latmn", "latss", # latitude as degrees, minutes, and seconds 
    "londd", "lonmn", "lonss", # longitude as degrees, minutes, and seconds 
    "hgt", "utmzone", "utm_e", "utm_n", "utm_s1", "utm_s2" # ellipsoidal height, UTM (universal transverse mercator aka repping location of Earth with flat (x,y) coords) zone number, how far east and north in meters, point scale factor and combined scale factor
]

'''Function to get one of the PPP files into a cleaned data frame of GPS epochs. Uses the backward and forward passes 
as position estimates to construct the decimal lat and long from the degree-minute-seconds column and apply a filter based on the position RMS to limit noise'''

def parse_one_pos(filepath, station=None):
    rows = []
    # Only keeping lines that start with a backward or forward pass 
    with open(filepath, 'r') as f:
        for line in f:
            if line.startswith("BWD") or line.startswith("FWD"):
                parts = line.split()
                if len(parts) >= 22:
                    rows.append(parts)
    if not rows:
        return None, None 
    
    #getting rid of columns that are not needed based on the slip catalog     
    raw_df = pd.DataFrame(rows)
    raw_df = raw_df.iloc[:, :len(column_labels)]
    raw_df.columns = column_labels[:raw_df.shape[1]]

    # Getting the timestamp from seperate date and time columns 
    raw_df["timestamp"] = pd.to_datetime(
        raw_df["date"] + " " + raw_df["time"],
        format="%Y-%m-%d %H:%M:%S.%f",
        errors="coerce"
    )

    # Numeric conversions from degree-minute-seconds 
    dms_columns = ["rmsp", "latdd", "latmn", "latss", "londd", "lonmn", "lonss", "hgt"]
    for col in dms_columns:
        if col in raw_df.columns:
            raw_df[col] = pd.to_numeric(raw_df[col], errors="coerce")

    # Going back to decimal degrees from the degree-minute-second format
    lat_sign = np.where(raw_df["latdd"] < 0, -1, 1)
    long_sign = np.where(raw_df["londd"] < 0, -1, 1)
    raw_df["lat"] = lat_sign * (raw_df["latdd"].abs() + raw_df["latmn"] / 60 + raw_df["latss"] / 3600)
    raw_df["lon"] = long_sign * (raw_df["londd"].abs() + raw_df["lonmn"] / 60 + raw_df["lonss"] / 3600)

    # Getting rid of epochs that have a lot of noise in their position based on the phase observation RMS
    pre_filter = raw_df[["timestamp", "lat", "lon", "hgt", "rmsp"]].copy()
    pre_filter["station"] = station

    raw_df = raw_df.dropna(subset=["timestamp", "lat", "lon"])
    raw_df = raw_df.sort_values("timestamp").reset_index(drop=True)

    return raw_df[["timestamp", "lat", "lon", "hgt", "rmsp"]], pre_filter

# Creating daily background/time series velocity 
def compute_station_velocity(station, all_files):
    # Load and concatenate all .pos files
    epoch_frames = []
    for fp in sorted(all_files):
        df, _ = parse_one_pos(fp)
        if df is not None and len(df) > 0:
            epoch_frames.append(df)

    if not epoch_frames:
        return None

    all_epochs = pd.concat(epoch_frames, ignore_index=True)
    all_epochs = all_epochs.drop_duplicates("timestamp").sort_values("timestamp").reset_index(drop=True)

    # Project all epochs to EPSG:3031
    east, north = proj_geo_to_polar(all_epochs["lat"].values, all_epochs["lon"].values)
    all_epochs["east"]  = east
    all_epochs["north"] = north

    # Using daily median positions since each position file has a 15 second measurement which is 5760 epochs which creates way too much noise
   
    all_epochs["date"] = all_epochs["timestamp"].dt.normalize()

    daily_positions = (all_epochs.groupby("date")
               .agg(
                   east  = ("east",  "median"),
                   north = ("north", "median"),
                   n     = ("east",  "count"),
                   rmsp  = ("rmsp",  "median"),
               )
               .reset_index()
               .rename(columns={"date": "timestamp"}))
# Require minimum number of good epochs per day (2 days)
    daily_positions = daily_positions[daily_positions["n"] >= min_epochs_day].copy()
    daily_positions["station"] = station

    if len(daily_positions) < 2:
        return None

    # displacement and time since previous days velocity 
    dt_sec = daily_positions["timestamp"].diff().dt.total_seconds()
    change_east = daily_positions["east"].diff()
    change_north = daily_positions["north"].diff()

    # If there are gaps, set those to Nan so they are not factored into velocity estimates 
    is_gap = (dt_sec > max_day_gap * 86400) | (dt_sec <= 0)
    change_east[is_gap] = np.nan
    change_north[is_gap] = np.nan
    dt_sec[is_gap] = np.nan
    
    #Getting magnitude from the components 
    daily_positions["vx"] = change_east / dt_sec * sec_per_year
    daily_positions["vy"] = change_north / dt_sec * sec_per_year
    daily_positions["v"]  = np.sqrt(daily_positions["vx"]**2 + daily_positions["vy"]**2)

    # Discarding velocities that don't make physical sense 
    for col in ["vx", "vy", "v"]:
        daily_positions.loc[daily_positions[col].abs() > phys_vel_threshold, col] = np.nan

    # Create a background velocity be using the timestamp as the index and then doing a 30-day rolling median
    daily_positions = daily_positions.set_index("timestamp").sort_index()
    for col in ["v", "vx", "vy"]:
        daily_positions[f"{col}_30d"] = (
            daily_positions[col]
            .rolling("30D", center=True, min_periods=5)
            .median()
        )
    # Put the timestamp back as a regular column     
    daily_positions = daily_positions.reset_index()

    return daily_positions
'''Parse all .pos files, collect raw fwd/bwd epochs before rmsp filtering,
    and stream-write to a single CSV with a 'station' column.
    Writing row-by-row avoids accumulating millions of epochs in RAM.'''
def build_pre_filter_csv():
    station_folder_names = sorted([
        d for d in os.listdir(PPP_dir)
        if os.path.isdir(os.path.join(PPP_dir, d))
    ])
    print(f"Pre-filter CSV: found {len(station_folder_names)} stations: {station_folder_names}")

    os.makedirs(os.path.dirname(Pre_filter_csv), exist_ok=True)
    header_written = False
    total_rows = 0

    for station_name in station_folder_names:
        all_pos_files = sorted(glob.glob(
            os.path.join(PPP_dir, station_name, "**", "*.pos"), recursive=True
        ))
        print(f"  {station_name}: {len(all_pos_files)} files", flush=True)
        station_rows = 0
        for i, fp in enumerate(all_pos_files):
            print(f"    [{i+1}/{len(all_pos_files)}] {os.path.basename(fp)}", end=" ", flush=True)
            _, pre = parse_one_pos(fp, station=station_name)
            if pre is not None and len(pre) > 0:
                pre.to_csv(Pre_filter_csv,
                           mode='w' if (not header_written) else 'a',
                           header=not header_written,
                           index=False)
                header_written = True
                station_rows += len(pre)
                total_rows   += len(pre)
                print(f"→ {len(pre)} rows", flush=True)
            else:
                print("→ empty", flush=True)
        print(f"  {station_name} done: {station_rows} rows written", flush=True)

    print(f"\nPre-filter CSV complete: {Pre_filter_csv}  total rows={total_rows}")
    # Return a lightweight confirmation dict rather than re-reading the whole file
    return {"path": Pre_filter_csv, "total_rows": total_rows}    
# Looping over every station and then concatinating it into one file 
def build_ppp_timeseries():
    # Grab all the station sub directories 
    station_folder_names = sorted([
        d for d in os.listdir(PPP_dir)
        if os.path.isdir(os.path.join(PPP_dir, d))
    ])
    print(f"Stations found: {station_folder_names}\n")

    station_timeseries = []

    # Recursively find all the position files for each station across all the sub folders 
    for station_name in station_folder_names:
        all_pos_files = sorted(glob.glob(
            os.path.join(PPP_dir, station_name, "**", "*.pos"), recursive=True
        ))
        print(f"  {station_name}: {len(all_pos_files)} files", end="", flush=True)
        #Constructing the time series 
        daily_ts = compute_station_velocity(station_name, all_pos_files)
        if daily_ts is None or len(daily_ts) == 0:
            print("no data")
            continue

        station_timeseries.append(daily_ts)
        #Printing so we can get a feel for where it's at in running 
        valid_v = daily_ts["v"].dropna()
        print(f" {len(daily_ts)} days"
              f"{daily_ts['timestamp'].min().date()} {daily_ts['timestamp'].max().date()} "
              f"median v = {valid_v.median():.1f} m/yr  "
              f"(n_valid={len(valid_v)})")

    full_ts = (pd.concat(station_timeseries, ignore_index=True)
                 .sort_values(["station", "timestamp"])
                 .reset_index(drop=True))

    full_ts.to_csv(Daily_timeseries_csv, index=False)
    print(f"\nSaved timeseries {Daily_timeseries_csv}  shape={full_ts.shape}")
    return full_ts

# Use the daily velocity estimates in a 30 day window to compute a median value for each station for each of the icequakes
def assign_velocities_to_slips(full_ts):
    # Get the icequake catalog 
    slips = pd.read_csv(Events_csv, parse_dates=["start_time"])
    print(f"\nEvents loaded: {len(slips)}")

    # Getting one dict for each event and station combo
    slip_station_records = []

    for station_name, station_daily in full_ts.groupby("station"):
        # Index by time
        station_daily = station_daily.set_index("timestamp").sort_index()

        for _, event_row in slips.iterrows():
            slip_time = event_row["start_time"]
            slip_start = slip_time - pd.Timedelta(days=pre_window_days)
            # Get all the daily estimates in the pre-event window
            pre_evt_window = station_daily.loc[slip_start:slip_time]

            # Skip if not enough events 
            if pre_evt_window["v"].notna().sum() < min_daily_pts:
                continue
            # Take the median and set up as one record per event (event, station)
            slip_station_records.append({
                "start_time":        slip_time,
                "station":           station_name,
                f"v_pre_{station_name}":  pre_evt_window["v"].median(),
                f"vx_pre_{station_name}": pre_evt_window["vx"].median(),
                f"vy_pre_{station_name}": pre_evt_window["vy"].median(),
                f"v30d_{station_name}":   pre_evt_window["v_30d"].median(),
                f"n_pre_{station_name}":  int(pre_evt_window["v"].notna().sum()),
            })

    if not slip_station_records:
        return slips
    vel_records_df = pd.DataFrame(slip_station_records)

    # Organizing like the tidal data so that each event is a row 
    vel_wide = (vel_records_df
                 .drop(columns="station")
                 .groupby("start_time")
                 .first()
                 .reset_index())
    # Put back into original events tabel 
    events_with_vel = slips.merge(vel_wide, on="start_time", how="left")
    events_with_vel.to_csv(Event_w_vel_csv, index=False)
    print(f"Saved {Event_w_vel_csv}  shape={events_with_vel.shape}")

    # Give some stats to make sure it is behaving as expected
    pre_vel_columns = sorted([c for c in events_with_vel.columns if c.startswith("v_pre_")])
    print("\nPre-event velocity summary")
    for col in pre_vel_columns:
        valid = events_with_vel[col].dropna()
        if len(valid) == 0:
            continue
        st = col.replace("v_pre_", "")
        print(f"  {st}: n={len(valid):4d} "
              f"mean={valid.mean():6.1f} "
              f"std={valid.std():5.1f} "
              f"min={valid.min():6.1f} "
              f"max={valid.max():6.1f} m/yr")

    return events_with_vel

if __name__ == "__main__":
    build_pre_filter_csv()
    full_ts = build_ppp_timeseries()
    events  = assign_velocities_to_slips(full_ts)

    print("\nDate range")
    print(f"  Events:     {events['start_time'].min().date()} → "
          f"{events['start_time'].max().date()}")
    print(f"  Timeseries: {full_ts['timestamp'].min().date()} → "
          f"{full_ts['timestamp'].max().date()}")

    print("\nCoverage: events with at least one velocity value")
    v_cols = [c for c in events.columns if c.startswith("v_pre_")]
    events["has_velocity"] = events[v_cols].notna().any(axis=1)
    print(f"  {events['has_velocity'].sum()} / {len(events)} events have velocity")

    print("\nPer-year coverage")
    events["year"] = events["start_time"].dt.year
    for year, grp in events.groupby("year"):
        n_total   = len(grp)
        n_covered = grp["has_velocity"].sum()
        stations  = [c.replace("v_pre_","") for c in v_cols
                     if grp[c].notna().any()]
        print(f"  {year}: {n_covered}/{n_total} events covered "
              f"stations: {stations}")
        


Pre-filter CSV: found 48 stations: ['gz01', 'gz02', 'gz03', 'gz04', 'gz05', 'gz06', 'gz07', 'gz08', 'gz09', 'gz10', 'gz11', 'gz12', 'gz13', 'gz14', 'gz15', 'gz16', 'gz17', 'gz18', 'gz19', 'gz20', 'la01', 'la02', 'la03', 'la04', 'la05', 'la06', 'la07', 'la08', 'la09', 'la10', 'la11', 'la12', 'la13', 'la14', 'la15', 'la16', 'la17', 'la18', 'mg01', 'mg02', 'mg03', 'mg04', 'mg05', 'mg06', 'mg07', 'slw1', 'ws04', 'ws05']
  gz01: 1067 files
    [1/1067] gz010110.pos → 5760 rows
    [2/1067] gz010120.pos → 5760 rows
    [3/1067] gz010130.pos → 5760 rows
    [4/1067] gz010140.pos → 5760 rows
    [5/1067] gz010150.pos → 5760 rows
    [6/1067] gz010160.pos → 5760 rows
    [7/1067] gz010170.pos → 5760 rows
    [8/1067] gz010180.pos → 5760 rows
    [9/1067] gz010190.pos → 5760 rows
    [10/1067] gz010200.pos → 5760 rows
    [11/1067] gz010210.pos → 5760 rows
    [12/1067] gz010220.pos → 5760 rows
    [13/1067] gz010230.pos → 5760 rows
    [14/1067] gz010240.pos → 5760 rows
    [15/1067] gz010250.p

KeyboardInterrupt: 